# SC-17-E2E-Verifiable-Voting - Vote Electronique Verifiable

[<< Homomorphic Encryption](SC-16-Homomorphic-Encryption.ipynb) | [Vyper >>](../05-Alternative-Chains/SC-18-Vyper.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre le **paradoxe du vote electronique** (anonymat + verifiabilite)
2. Construire un systeme de vote **a la main** avec Paillier + ZKP
3. Implementer un **bulletin board** publiquement verifiable
4. Decouvrir **ElectionGuard** (Microsoft), le SOTA en vote E2E verifiable
5. Relier ces techniques aux smart contracts

### Prerequis

- [SC-15](SC-15-Zero-Knowledge-Proofs.ipynb) et [SC-16](SC-16-Homomorphic-Encryption.ipynb) completes
- `phe` (python-paillier) installe
- `pycryptodome` installe
- `electionguard` installe (optionnel, pour la partie SOTA)

### Duree estimee : 70 minutes

---

## 1. Le probleme du vote electronique

Le vote electronique doit satisfaire des proprietes **contradictoires en apparence** :

| Propriete | Description | Tension |
|-----------|-------------|--------|
| **Anonymat** | Personne ne sait ce que j'ai vote | vs Verifiabilite |
| **Verifiabilite individuelle** | Je peux verifier que mon vote a ete compte | vs Anonymat |
| **Verifiabilite universelle** | N'importe qui peut verifier le decompte | vs Anonymat |
| **Eligibilite** | Seuls les electeurs autorises votent | vs Anonymat |
| **Unicite** | Un electeur = un vote | vs Anonymat |
| **Resistance a la coercition** | Impossible de prouver pour qui on a vote | vs Verifiabilite |

La solution : le **chiffrement homomorphique** permet de compter les votes
sans les dechiffrer individuellement, et les **preuves a divulgation nulle** (ZKP)
permettent de verifier qu'un bulletin est valide sans reveler son contenu.

### Historique

| Annee | Systeme | Contribution |
|-------|---------|-------------|
| 2008 | **Helios** (Ben Adida) | Premier systeme E2E verifiable grand public |
| 2012+ | **Belenios** (INRIA) | Preuves formelles en Coq, plus fort garanties |
| 2019 | **ElectionGuard** (Microsoft) | SDK Python production-ready, threshold decryption |

---

## 2. Chiffrement des bulletins (Paillier)

Chaque electeur chiffre son vote avec Paillier (additively homomorphic).
Un vote pour le candidat i est represente comme un vecteur one-hot :
- Candidat A : `[1, 0, 0]`
- Candidat B : `[0, 1, 0]`
- Candidat C : `[0, 0, 1]`

Chaque composante est chiffree individuellement.

In [ ]:
from phe import paillier
import hashlib
import json

# Autorite electorale : genere les cles
public_key, private_key = paillier.generate_paillier_keypair(n_length=1024)
print("AUTORITE ELECTORALE")
print("=" * 60)
print(f"Cle publique (n) : {str(public_key.n)[:40]}...")
print(f"Cle publique distribuee a tous les electeurs")
print(f"Cle privee conservee par l'autorite (ou repartie via threshold)")
print()

# Candidats
candidates = ["Alice Dupont", "Bob Martin", "Charlie Durand"]
print(f"Candidats : {candidates}")
print()

In [ ]:
# Simulation de 7 electeurs
import random
random.seed(42)  # reproductibilite

voters = [
    {"name": "Electeur 1", "choice": 0},  # vote Alice
    {"name": "Electeur 2", "choice": 1},  # vote Bob
    {"name": "Electeur 3", "choice": 0},  # vote Alice
    {"name": "Electeur 4", "choice": 2},  # vote Charlie
    {"name": "Electeur 5", "choice": 0},  # vote Alice
    {"name": "Electeur 6", "choice": 1},  # vote Bob
    {"name": "Electeur 7", "choice": 2},  # vote Charlie
]

# Chaque electeur chiffre son bulletin
encrypted_ballots = []
print("CHIFFREMENT DES BULLETINS")
print("=" * 60)

for voter in voters:
    # Creer le vecteur one-hot
    plaintext = [0] * len(candidates)
    plaintext[voter["choice"]] = 1

    # Chiffrer chaque composante
    encrypted = [public_key.encrypt(v) for v in plaintext]
    encrypted_ballots.append(encrypted)

    # Le bulletin chiffre ne revele rien
    print(f"  {voter['name']} : bulletin chiffre (3 ciphertexts de ~{len(str(encrypted[0].ciphertext()))} digits)")

print(f"\n-> {len(encrypted_ballots)} bulletins chiffres, aucun vote individuel visible")

### Interpretation

Chaque bulletin chiffre est un vecteur de 3 ciphertexts Paillier.
Meme l'autorite ne peut pas deviner le contenu d'un bulletin individuel
sans utiliser la cle privee. Le chiffrement est **semantiquement securise** :
deux chiffrements du meme message sont differents (grace a l'aleatoire).

---

## 3. Decompte homomorphique

Grace a la propriete additive de Paillier, on peut **additionner les bulletins chiffres**
sans les dechiffrer. Seul le **total** est ensuite dechiffre.

In [ ]:
# Addition homomorphique des bulletins
print("DECOMPTE HOMOMORPHIQUE")
print("=" * 60)

# Initialiser les totaux chiffres a 0
tally_encrypted = [public_key.encrypt(0) for _ in candidates]

# Additionner tous les bulletins
for ballot in encrypted_ballots:
    for i in range(len(candidates)):
        tally_encrypted[i] = tally_encrypted[i] + ballot[i]

print("Totaux chiffres calcules (sans dechiffrer aucun bulletin individuel)")
print()

# Seul le total est dechiffre par l'autorite
results = [private_key.decrypt(t) for t in tally_encrypted]

print("RESULTATS :")
print("-" * 40)
for i, (candidate, votes) in enumerate(zip(candidates, results)):
    bar = "#" * votes
    print(f"  {candidate:20s} : {votes} votes  {bar}")
print(f"  {'Total':20s} : {sum(results)} votes")
print()

# Verification manuelle
expected = [0] * len(candidates)
for v in voters:
    expected[v["choice"]] += 1
assert results == expected, "Le decompte homomorphique doit correspondre au decompte manuel"
print("-> Verification : le decompte homomorphique correspond au decompte en clair")

### Interpretation

Le decompte s'est fait **entierement sur des donnees chiffrees** :
- Aucun bulletin individuel n'a ete dechiffre
- Seul le resultat agrege a ete revele
- L'anonymat de chaque electeur est preserve

Mais comment garantir que les bulletins sont **valides** (exactement un 1 et des 0) ?
C'est la que les ZKP interviennent.

---

## 4. Preuve de validite du bulletin (ZKP simplifiee)

Chaque electeur doit prouver que son bulletin chiffre contient un **vote valide**
(exactement un 1 parmi les composantes) **sans reveler pour qui il a vote**.

Nous implementons ici une version simplifiee basee sur la **somme des composantes**.
Un bulletin valide a une somme de composantes = 1 (un seul candidat choisi).

In [ ]:
import hashlib

def compute_ballot_hash(encrypted_ballot):
    """Calculer un identifiant unique pour un bulletin chiffre."""
    data = "|".join(str(e.ciphertext()) for e in encrypted_ballot)
    return hashlib.sha256(data.encode()).hexdigest()

def prove_valid_ballot(public_key, encrypted_ballot, plaintext_vote, num_candidates):
    """Generer une preuve simplifiee que le bulletin est valide.

    Preuve que sum(composantes) = 1 en utilisant la propriete homomorphique.
    Le verifieur peut additionner les ciphertexts et verifier que le resultat
    dechiffre a 1, MAIS sans connaitre la cle privee. On utilise donc un
    commitment scheme.
    """
    # La preuve contient :
    # 1. La somme chiffree des composantes
    sum_encrypted = encrypted_ballot[0]
    for e in encrypted_ballot[1:]:
        sum_encrypted = sum_encrypted + e

    # 2. Un hash du bulletin pour identification
    ballot_hash = compute_ballot_hash(encrypted_ballot)

    return {
        "ballot_hash": ballot_hash,
        "sum_encrypted": sum_encrypted,
        "num_candidates": num_candidates,
    }

def verify_ballot_proof(private_key, proof):
    """Verifier la preuve (par l'autorite).

    Note: dans un vrai systeme, la verification se fait sans cle privee
    via des preuves Sigma / Chaum-Pedersen. Ici on simplifie.
    """
    decrypted_sum = private_key.decrypt(proof["sum_encrypted"])
    return decrypted_sum == 1


# Generer et verifier les preuves
print("PREUVES DE VALIDITE DES BULLETINS")
print("=" * 60)

all_valid = True
proofs = []
for i, (voter, ballot) in enumerate(zip(voters, encrypted_ballots)):
    proof = prove_valid_ballot(public_key, ballot, voter["choice"], len(candidates))
    valid = verify_ballot_proof(private_key, proof)
    proofs.append(proof)
    status = "VALIDE" if valid else "INVALIDE"
    print(f"  {voter['name']} : {status} (sum=1, ballot_hash={proof['ballot_hash'][:16]}...)")
    if not valid:
        all_valid = False

print(f"\n-> Tous les bulletins valides : {all_valid}")

# Demonstration : un bulletin invalide serait detecte
print("\nDEMONSTRATION : bulletin invalide")
fake_ballot = [public_key.encrypt(1), public_key.encrypt(1), public_key.encrypt(0)]  # 2 votes !
fake_proof = prove_valid_ballot(public_key, fake_ballot, 0, len(candidates))
fake_valid = verify_ballot_proof(private_key, fake_proof)
print(f"  Bulletin avec 2 votes : {'VALIDE' if fake_valid else 'REJETE'}")
print("  -> Un electeur ne peut pas voter deux fois dans un bulletin")

### Interpretation

Dans notre version simplifiee, l'autorite verifie les preuves avec sa cle privee.
Dans un vrai systeme (Helios, ElectionGuard), les preuves sont des **ZKP non-interactives**
(Chaum-Pedersen / Sigma protocols) verifiables par **n'importe qui** sans cle privee.

C'est la difference entre notre demo pedagogique et un systeme production-ready.

---

## 5. Bulletin board public

Le **bulletin board** est un registre public ou tous les bulletins chiffres
et leurs preuves sont publies. N'importe qui peut :
- Verifier que son bulletin est present
- Verifier les preuves de validite de tous les bulletins
- Recompter le total homomorphique

In [ ]:
# Construction du bulletin board
bulletin_board = []
for i, (voter, ballot, proof) in enumerate(zip(voters, encrypted_ballots, proofs)):
    entry = {
        "voter_id": hashlib.sha256(voter["name"].encode()).hexdigest()[:16],
        "ballot_hash": proof["ballot_hash"],
        "proof_valid": verify_ballot_proof(private_key, proof),
        "encrypted_ballot": ballot,  # les ciphertexts
    }
    bulletin_board.append(entry)

print("BULLETIN BOARD PUBLIC")
print("=" * 60)
print("  Voter ID           | Ballot Hash      | Proof")
print("-" * 60)
for entry in bulletin_board:
    status = "OK" if entry["proof_valid"] else "FAIL"
    vid = entry["voter_id"]
    bh = entry["ballot_hash"][:16]
    print(f"  {vid:16s} | {bh}... | {status}")

print(f"\nTotal : {len(bulletin_board)} bulletins publies")

In [ ]:
# Verification individuelle : un electeur verifie que son bulletin est present
print("VERIFICATION INDIVIDUELLE")
print("=" * 60)

# L'electeur 3 verifie
my_voter = voters[2]
my_ballot = encrypted_ballots[2]
my_hash = compute_ballot_hash(my_ballot)

# Chercher dans le bulletin board
found = False
for entry in bulletin_board:
    if entry["ballot_hash"] == my_hash:
        found = True
        break

print(f"Electeur : {my_voter['name']}")
print(f"Mon ballot hash : {my_hash[:32]}...")
print(f"Present dans le bulletin board : {'OUI' if found else 'NON'}")
print()
print("-> L'electeur peut verifier que son vote a bien ete enregistre")
print("-> MAIS il ne peut pas prouver a un tiers pour qui il a vote")
print("   (le hash ne revele pas le contenu, et le chiffrement est randomise)")

### Interpretation

Le bulletin board fournit la **transparence** :

| Propriete | Mecanisme |
|-----------|----------|
| Verifiabilite individuelle | L'electeur conserve le hash de son bulletin |
| Verifiabilite universelle | Quiconque peut reverifier le decompte homomorphique |
| Anonymat | Le hash ne revele pas le vote, le chiffrement est randomise |
| Resistance a la coercition | L'electeur ne peut pas prouver son vote a un tiers |

Ce modele est exactement celui utilise par Helios (2008) et ses successeurs.

---

## 6. ElectionGuard - Le SOTA

**ElectionGuard** (Microsoft, 2019) est le systeme de vote E2E verifiable
le plus avance, disponible en open-source avec un SDK Python.

### Differences avec notre construction a la main

| Aspect | Notre demo | ElectionGuard |
|--------|-----------|---------------|
| Chiffrement | Paillier (additif) | ElGamal exponentiel (additif + ZKP) |
| ZKP bulletins | Verification avec cle privee | Chaum-Pedersen non-interactive |
| Dechiffrement | Cle privee unique | Threshold (k-of-n gardiens) |
| Bulletin board | Liste Python | Artefacts JSON standardises |
| Audit | Manuel | Outils de verification automatises |

### Architecture ElectionGuard

```
1. Configuration : Manifest (candidats, regles) + Gardiens (threshold keys)
2. Chiffrement   : Chaque bulletin chiffre avec ElGamal + ZKP Chaum-Pedersen
3. Publication    : Bulletin board avec tous les ciphertexts + preuves
4. Decompte      : Addition homomorphique des ciphertexts
5. Dechiffrement : k-of-n gardiens revelent le total (threshold decryption)
6. Verification  : Quiconque peut reverifier avec les artefacts publics
```

### Threshold Decryption

La cle privee est **repartie entre N gardiens** (ex: 5 commissaires electoraux).
Il faut au moins **k gardiens** (ex: 3) pour dechiffrer le resultat.
Aucun gardien seul ne peut dechiffrer quoi que ce soit.

In [ ]:
# ElectionGuard SDK (optionnel - necessite pip install electionguard)
try:
    from electionguard.election import CiphertextElectionContext
    print("ElectionGuard installe et disponible")
    ELECTIONGUARD_AVAILABLE = True
except ImportError:
    print("ElectionGuard non installe.")
    print("Pour l'installer : pip install electionguard")
    print()
    print("Note : ElectionGuard necessite pydantic<2 (conflit avec web3>=7)")
    print("Installer dans un venv separe si necessaire.")
    ELECTIONGUARD_AVAILABLE = False
except Exception as e:
    # ElectionGuard a des bugs internes avec Python 3.12+ (mutable dataclass defaults)
    print(f"ElectionGuard installe mais incompatible avec ce Python : {type(e).__name__}")
    print(f"  {e}")
    print()
    print("La section suivante montre le code ElectionGuard a titre d'illustration.")
    ELECTIONGUARD_AVAILABLE = False

In [ ]:
# Code ElectionGuard illustratif
# (s'execute uniquement si la library est installee)

if ELECTIONGUARD_AVAILABLE:
    # Cet exemple simplifie est base sur la documentation officielle
    # https://www.electionguard.vote/
    print("Configuration d'une election ElectionGuard...")
    print("(voir la documentation officielle pour un exemple complet)")
else:
    # Simuler la structure conceptuelle
    print("SIMULATION CONCEPTUELLE ElectionGuard")
    print("=" * 60)
    print()
    print("1. MANIFEST (definition de l'election) :")
    manifest = {
        "election_scope_id": "election-municipale-2026",
        "type": "general",
        "candidates": ["Alice Dupont", "Bob Martin", "Charlie Durand"],
        "contests": [{"id": "maire", "type": "plurality", "max_selections": 1}],
        "guardians": {"count": 5, "quorum": 3},
    }
    for k, v in manifest.items():
        print(f"   {k}: {v}")

    print()
    print("2. CEREMONY DES GARDIENS (generation des cles threshold) :")
    print("   5 gardiens generent chacun une paire de cles")
    print("   Ils partagent des fragments de cle (Shamir secret sharing)")
    print("   La cle publique jointe est publiee")
    print("   -> Quorum de 3/5 necessaire pour dechiffrer")

    print()
    print("3. CHIFFREMENT DES BULLETINS :")
    print("   Chaque bulletin chiffre avec ElGamal exponentiel")
    print("   + preuve Chaum-Pedersen de validite (0 ou 1 par candidat)")
    print("   + preuve que la somme des selections = 1")

    print()
    print("4. DECOMPTE :")
    print("   Addition homomorphique des ciphertexts ElGamal")
    print("   Dechiffrement par 3+ gardiens (threshold)")
    print("   Publication des preuves de dechiffrement correct")

    print()
    print("5. VERIFICATION E2E :")
    print("   N'importe qui telecharge les artefacts et verifie :")
    print("   - Toutes les preuves de bulletin sont valides")
    print("   - Le decompte correspond a la somme des ciphertexts")
    print("   - Le dechiffrement est correct (preuve DLOG)")

### Comparaison historique

| Critere | Helios (2008) | Belenios (2012+) | ElectionGuard (2019) |
|---------|--------------|-----------------|---------------------|
| Chiffrement | ElGamal | ElGamal | ElGamal |
| ZKP | Sigma protocols | Sigma protocols | Chaum-Pedersen |
| Preuves formelles | Non | Oui (Coq) | Non (mais audite) |
| Threshold | Non (trustee unique) | Oui | Oui (k-of-n) |
| SDK | JavaScript | OCaml | **Python** |
| Usage reel | Elections universitaires | Elections CNRS | Pilotes US |
| Open source | Oui | Oui | Oui (Microsoft) |

ElectionGuard est le plus accessible grace a son **SDK Python** et sa documentation complete.

---

## 7. Exercice : Vote prefrentiel verifiable

Etendez le systeme de vote pour supporter le **vote prefrentiel** (ranked-choice)
tout en conservant les proprietes de confidentialite et verifiabilite.

### Principe du vote prefrentiel

Au lieu de voter pour un seul candidat, chaque electeur **classe** les candidats
par ordre de preference. Si aucun candidat n'a la majorite absolue au premier tour,
le candidat avec le moins de votes de premier choix est elimine, et ses votes
sont redistribues selon les seconds choix, etc.

### Defi cryptographique

Le decompte n'est plus une simple somme : il faut des **tours iteratifs**
avec elimination. Comment faire cela sur des bulletins chiffres ?

In [ ]:
# Exercice : Vote prefrentiel chiffre

from phe import paillier
import hashlib

# Configuration
candidates_ex = ["Alice", "Bob", "Charlie"]
pk, sk = paillier.generate_paillier_keypair(n_length=1024)


class RankedBallot:
    """Bulletin de vote prefrentiel chiffre.

    Representation : matrice N x N ou entry[i][j] = 1 si le candidat i
    est classe en position j (0-indexed), 0 sinon.
    Chaque ligne a exactement un 1, chaque colonne aussi (permutation).
    """

    def __init__(self, public_key, ranking):
        """Creer un bulletin chiffre a partir d'un classement.

        Args:
            public_key: cle publique Paillier
            ranking: liste d'indices de candidats par preference
                     ex: [2, 0, 1] = Charlie 1er, Alice 2eme, Bob 3eme
        """
        self.n_candidates = len(ranking)
        # TODO: Construire la matrice de permutation
        # TODO: Chiffrer chaque element de la matrice
        # self.encrypted_matrix = [[pk.encrypt(...) for j in ...] for i in ...]
        raise NotImplementedError("Implementez le chiffrement du bulletin prefrentiel")

    def get_first_choice_vector(self):
        """Retourner le vecteur des premiers choix (colonne 0 de la matrice).
        TODO: Extraire la colonne 0 de encrypted_matrix
        """
        raise NotImplementedError("Implementez l'extraction du premier choix")

    def get_choice_at_rank(self, rank):
        """Retourner le vecteur des choix au rang donne.
        TODO: Extraire la colonne 'rank' de encrypted_matrix
        """
        raise NotImplementedError("Implementez l'extraction d'un rang")


def ranked_tally(ballots, private_key, candidates):
    """Decompte par elimination successive (Instant Runoff Voting).

    TODO:
    1. Premier tour : additionner les vecteurs de premier choix
    2. Dechiffrer le total pour trouver le candidat avec le moins de votes
    3. Eliminer ce candidat
    4. Pour les bulletins dont le premier choix est elimine,
       utiliser le second choix (et ainsi de suite)
    5. Repeter jusqu'a ce qu'un candidat ait la majorite

    Note : cette operation necessite de dechiffrer les totaux intermediaires,
    mais PAS les bulletins individuels.
    """
    raise NotImplementedError("Implementez le decompte prefrentiel")


# Test (decommentez apres implementation)
# ballots = [
#     RankedBallot(pk, [0, 1, 2]),  # Alice > Bob > Charlie
#     RankedBallot(pk, [1, 0, 2]),  # Bob > Alice > Charlie
#     RankedBallot(pk, [2, 0, 1]),  # Charlie > Alice > Bob
#     RankedBallot(pk, [0, 2, 1]),  # Alice > Charlie > Bob
#     RankedBallot(pk, [2, 1, 0]),  # Charlie > Bob > Alice
# ]
# winner = ranked_tally(ballots, sk, candidates_ex)
# print(f"Gagnant : {candidates_ex[winner]}")

---

## 8. Resume

| Composant | Technique | Role |
|-----------|-----------|------|
| **Anonymat** | Chiffrement Paillier / ElGamal | Personne ne voit les votes individuels |
| **Decompte** | Addition homomorphique | Compter sans dechiffrer |
| **Validite** | Preuves ZKP (Chaum-Pedersen) | Verifier qu'un bulletin est bien forme |
| **Transparence** | Bulletin board public | Tout le monde peut verifier |
| **Resilience** | Threshold decryption (k-of-n) | Aucun gardien seul ne peut dechiffrer |
| **Verification** | Artefacts publics | N'importe qui peut recompter |

### Points cles

- Le vote E2E verifiable resout le paradoxe anonymat/verifiabilite grace a la cryptographie
- Le chiffrement homomorphique permet le **decompte sur des donnees chiffrees**
- Les ZKP garantissent la **validite des bulletins** sans reveler leur contenu
- ElectionGuard (Microsoft, 2019) est le SOTA avec un SDK Python
- L'integration blockchain (bulletin board sur Ethereum) ajoute l'**immutabilite**

---

**Notebook suivant** : [SC-18-Vyper](../05-Alternative-Chains/SC-18-Vyper.ipynb) - Smart contracts en Python-like

---

[<< Homomorphic Encryption](SC-16-Homomorphic-Encryption.ipynb) | [Vyper >>](../05-Alternative-Chains/SC-18-Vyper.ipynb)